# System Recommenders - Final Project 2025

### 🎯 Objective

Develop a recommender system that suggests short videos to users based on user preferences, interaction histories, and video content using the KuaiRec dataset. 

The challenge is to create a personalised and scalable recommendation engine similar to those used in platforms like TikTok or Kuaishou.

### 📥 Imports

In [5]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import csr_matrix
from implicit.als import AlternatingLeastSquares

plt.rcParams["figure.figsize"] = (20, 13)
%matplotlib inline
%config InlineBackend.figure_format = "retina"

### 📊 Download Dataset

We will use the **KuaiRec dataset**, a large-scale, fully-observed dataset collected from the Kuaishou short-video platform.

It contains:

- **User interactions** (views, likes, etc.)
- **Video metadata** (video ID, tags, etc.)
- **Timestamps**

More info: [KuaiRec Paper](https://arxiv.org/abs/2202.10842)

**Download dataset**

1. <ins>First option : Downloading Dataset via wget<ins>

In [6]:
%%bash
if [ ! -d "./data_final_project" ]; then
  wget --no-check-certificate 'https://drive.usercontent.google.com/download?id=1qe5hOSBxzIuxBb1G_Ih5X-O65QElollE&export=download&confirm=t&uuid=b2002093-cc6e-4bd5-be47-9603f0b33470' -O KuaiRec.zip
  unzip KuaiRec.zip -d ../data_final_project
else
  echo "Directory './data_final_project' already exists. Skipping download."
fi

Directory './data_final_project' already exists. Skipping download.


2. <ins>Second option : Downloading dataset via Google Drive<ins>

If the data is not downloaded by the wget because of a Connection Refused you might download it via this [link](https://drive.google.com/file/d/1qe5hOSBxzIuxBb1G_Ih5X-O65QElollE/view) and place it at the project root.

In [7]:
""" Uncomment if you need to (and if the first option did not work correctly)
%%bash
unzip KuaiRec.zip -d data_final_project
mv "data_final_project/KuaiRec 2.0/" data_final_project/KuaiRec
"""

' Uncomment if you need to (and if the first option did not work correctly)\n%%bash\nunzip KuaiRec.zip -d data_final_project\nmv "data_final_project/KuaiRec 2.0/" data_final_project/KuaiRec\n'

From this dataset we obtain the following files :

```bash
KuaiRec
  ├── data
  │   ├── big_matrix.csv          
  │   ├── small_matrix.csv
  │   ├── social_network.csv
  │   ├── user_features.csv
  │   ├── item_daily_features.csv
  │   └── item_categories.csv
  │   └── kuairec_caption_category.csv
```

- `interactions_train.csv`: historical user-item interactions for training.
- `interactions_test.csv`: user-item pairs to score during testing.
- `sample_submission.csv`: a template showing the expected output format.
- `video_metadata.csv`: metadata including tags or content-related features.

![image](img/KuaiRec.png)

## **1️⃣ Dataset Preprocessing**
📝 Associated Tasks :
- Load and inspect the dataset.
- Handle missing or inconsistent data.
- Merge metadata for content-based models if necessary.

---


<div style="background-color:#cccccc; padding: 10px; border-radius: 5px; border: 1px solid #aaa; font-weight: bold; color:#111;">
    💡 For more details about the different analyses and pre-processing decisions made on the available datasets, <a href="../EDA/EDA.ipynb" style="color:#1a73e8;">feel free to click here and go to the EDA notebook</a>.
</div>


---

However, here is a small recap of the observations made : 

- **User Interactions**: 
  - The majority of users have **fewer than 3,000 interactions**.
  - There are a few **outliers** with interactions exceeding 6,000.

- **Item Popularity**:
  - There are very few items that receive **high levels of interaction**.
  - Many items, while not extremely popular, still receive a fair amount of attention.
  - A small subset of items are truly **highly popular**.

- **Time-Based Trends**:
  - However, interactions peak towards the **end of the week**, specifically from **Friday to Sunday**.
  - **Late-night hours (0:00 - 3:00 a.m.)** see the highest levels of activity.
  - **From 10:00 to 20:00** we see less activity
  - The time of day or day of the week has **minimal to no impact** on the watch ratio.

- **User Behavior**:
  - The **top 10 most active users** show a distinct dip in activity around **12:00 p.m.** and **4:00 p.m.**.

- **Video Length**:
  - **Shorter videos** (up to 30 seconds) receive **more interactions** and have a **higher watch ratio**.
  - Conversely, **longer videos** tend to have **fewer interactions** and a lower watch ratio.
  - Videos that are **no longer than 30 seconds** appear to have the optimal watch ratio.

- **Watch Ratio**:
  - **Half of the videos have less then 75% watch ratio.**
  - Only **0.1% of the videos have more than 18** of watch ratio. These are extreme outliers, we can either drop them or try and normalize them.

- **Video Type:**

    - **"Ad" videos have significantly lower interaction** rates compared to Normal videos, which are more engaging for users.
    - Users tend to engage less with promotional or advertisement-type content.

- **User Preferences:**

    - There is a clear preference for **Short Imports videos**. These videos are viewed and interacted with more often compared to longer or different upload types.

- **Video Format:**

    - The **1280x720 resolution is the optimal format** for maximizing user interaction and watch ratio. Other formats either fall short or show no significant improvement.

- **Video Privacy Settings:**

    - As expected, **public videos receive more interactions** than private or only friends videos. This could be due to the higher volume of public content being uploaded, which in turn increases the overall interactions.

    - Private or only friends videos tend to have a smaller, more targeted audience, reducing their interaction rates.

- **Video Age**:
    - **Recent videos (lower video age in days) tend to have more interactions**



### Load Datasets

Here as we have shown in the [EDA notebook](./EDA/EDA.ipynb) with more details :

1. We have to load the dataset we will use.

    We will only be using three datasets :
    - `big_matrix.csv`
        - Corresponds to the different interactions made on the videos
        - It will serve for the training
    - `small_matrix.csv`
        - Same as big_matrix
        - It will serve for the testing
    - `item_daily_features.csv`
        - Contains informations about the video (e.g. number of likes, shares, reports ...) 
        - It will be merged both with the small and big matrix to give additional informations

2. We make small preprocessing :
    - Converting some columns to a more appropriate type
    - Removing highly correlated columns
    - We will cap videos with an above 2.34 watch ratio as above are only the top 5% as show in the [EDA notebook](./EDA/EDA.ipynb). This will help normalize the `watch_ratio` which is an important metric.


In [8]:
interactions = pd.read_csv("./data_final_project/KuaiRec/data/big_matrix.csv")
small_interactions = pd.read_csv("./data_final_project/KuaiRec/data/small_matrix.csv")
item_features = pd.read_csv("./data_final_project/KuaiRec/data/item_daily_features.csv")

def clean_df(df):
    df = df.dropna()
    df = df.drop_duplicates()  
    return df  

def clean_df_timestamp(df):
    df = clean_df(df)
    df = df[df["timestamp"] >= 0]
    df = df[df["watch_ratio"] <= 200]
    return df

item_features = clean_df(item_features)
item_features = item_features.drop_duplicates(subset='video_id')
train_df = clean_df_timestamp(interactions)
test_df = clean_df_timestamp(small_interactions)

item_features['upload_dt'] = pd.to_datetime(item_features['upload_dt'])
item_features['date'] = pd.to_datetime(item_features['date'], format='%Y%m%d')


We drop some columns in the test and train dataframes that we will not need.

In [9]:
to_drop = ['date', 'play_duration', 'video_duration', 'time', 'timestamp']

train_df.drop(columns=to_drop, inplace=True, errors='ignore')
test_df.drop(columns=to_drop, inplace=True, errors='ignore')

We apply a cap to 2.34 to the watch ratio, to prevent outliers.

This decision is mostly made to not only suggest videos that have a way too high watch ratio 
(some of the videos have a 600 watch ratio) but also some videos that have a smaller watch ratio.

In [10]:
train_df['watch_ratio'] = train_df['watch_ratio'].apply(lambda x: min(x, 2.34))
test_df['watch_ratio'] = test_df['watch_ratio'].apply(lambda x: min(x, 2.34))

We drop columns that have high correlation between them.

In [11]:
correlation = item_features[[
       'video_duration', 'video_width',
       'video_height', 'music_id',
       'show_cnt', 'show_user_num', 'play_cnt', 'play_user_num',
       'play_duration', 'complete_play_cnt', 'complete_play_user_num',
       'valid_play_cnt', 'valid_play_user_num', 'long_time_play_cnt',
       'long_time_play_user_num', 'short_time_play_cnt',
       'short_time_play_user_num', 'play_progress', 'comment_stay_duration',
       'like_cnt', 'like_user_num', 'click_like_cnt', 'double_click_cnt',
       'cancel_like_cnt', 'cancel_like_user_num', 'comment_cnt',
       'comment_user_num', 'direct_comment_cnt', 'reply_comment_cnt',
       'delete_comment_cnt', 'delete_comment_user_num', 'comment_like_cnt',
       'comment_like_user_num', 'follow_cnt', 'follow_user_num',
       'cancel_follow_cnt', 'cancel_follow_user_num', 'share_cnt',
       'share_user_num', 'download_cnt', 'download_user_num', 'report_cnt',
       'report_user_num', 'reduce_similar_cnt', 'reduce_similar_user_num',
       'collect_cnt', 'collect_user_num', 'cancel_collect_cnt',
       'cancel_collect_user_num']].corr()

upper = correlation.where(np.triu(np.ones(correlation.shape), k=1).astype(bool))

to_drop = [column for column in upper.columns if any(upper[column] > 0.8)]
# These columns are dropped as we don't need them anymore 
# (merged in previous cell or just not needed for collaborative-filtering)
item_features.drop(columns=to_drop, inplace=True, errors='ignore')


## **2️⃣ Feature Engineering**
📝 Associated Tasks :
- Create meaningful features from interaction and metadata (e.g., content tags, user activity history).
- Build user-item interaction matrix.
- Optionally extract time-based or popularity-based features.

---

### Basic additional features

As suggested, we will now create some new features for the datasets, the first two will correspond respectively to the `video_age` and a boolean `is_short_video` to see if the video is shorter then 30 seconds because as we saw in the [EDA notebook](./EDA/EDA.ipynb), shorter and recent videos are more interacted with then other type of videos.

In [12]:
item_features['video_age'] = (item_features['date'] - item_features['upload_dt']).dt.days
item_features['is_short_video'] = (item_features['video_duration'].fillna(0) <= 30).astype(int)

We remove those columns as we don't need them in the future.

In [13]:
to_drop =   [
                'date', 'upload_dt', 'video_duration', 'music_id',
                'video_tag_name', 'play_progress', 'video_tag_id',
                'time', 'play_duration'
            ]
item_features.drop(columns=to_drop, inplace=True, errors='ignore')

Let's now merge the item_features dataset, containing some insights on the videos such as number of likes, to our train_df and test_df.

In [14]:
train_df = pd.merge(train_df, item_features, on='video_id', how='left')
test_df = pd.merge(test_df, item_features, on='video_id', how='left')

### Engagement Score

To effectively train our ALS (Alternating Least Squares) recommendation model, we need a **proxy rating** to represent user interest or satisfaction with a video. 

We thus can define a **custom engagement score** that combines multiple implicit feedback signals. 

Based on insights from our exploratory data analysis ([see EDA notebook](./EDA/EDA.ipynb)), we define the **engagement score** using a combination of behavioral, content-related, and video metadata features.

These features capture how likely a user is to have positively interacted with a video, making this score a suitable proxy for collaborative filtering.

#### Key features used to compute the engagement score:

- **Watch Ratio**: Proportional score based on the ratio of the video watched. A higher watch ratio directly increases the score.
- **Short Videos (`is_short_video`)**: Users show a preference for short videos, so these receive a small bonus.
- **Video Age**: Newer videos are generally more engaging; the score includes a time decay bonus favoring recent uploads.
- **Video Type**: Videos marked as **"AD"** receive a penalty.
- **Upload Type**: Some formats are known to drive higher engagement (e.g., `"ShortImport"`, `"StartCamera"`), and are weighted accordingly.
- **Visibility Status**: Public videos are more likely to be seen and engaged with, hence receive a bonus.
- **Video Resolution**: The preferred format is `1280x720`. Videos with this resolution or higher receive more interactions.
- **User Interaction Metrics**:
  - Positive interactions (`like_cnt`, `comment_cnt`, `share_cnt`, etc.) are weighted logarithmically to contribute to the score.
  - Negative feedback (`cancel_like_cnt`, `cancel_follow_cnt`, `report_cnt`) reduces the score, also using logarithmic scaling.

Finally, the score is **min-max normalized** between `0` and `1` to ensure comparability across the dataset and improve model convergence.

This engagement score becomes our **implicit rating** that the ALS model will learn to predict.


In [ ]:
def build_engagement_score(df):
    df["engagement_score"] = 0
    
    # Watch Ratio
    if 'watch_ratio' in df.columns:
        df["engagement_score"] += df['watch_ratio'].fillna(0) * 10
    
    # is_short_video
    df["engagement_score"] += df['is_short_video'].fillna(0) * 3
    
    # Video age
    if 'video_age' in df.columns:
        max_age = 365
        normalized_age = np.minimum(df['video_age'].fillna(max_age), max_age) / max_age
        # Newer videos get up to 2 points bonus
        df["engagement_score"] += (1 - normalized_age) * 2
    
    
    if 'video_type' in df.columns:
        df["engagement_score"] += np.where(
            df['video_type'] == 'AD',
            -3,  # penalty for ads
            2    # bonus for regular content
        )
    
    if 'visible_status' in df.columns:
        df["engagement_score"] += np.where(
            df['visible_status'] == 'public',
            2,  
            -1
        )
    
    if 'upload_type' in df.columns:
        upload_type_weights = {
            'ShortImport': 3,     # Short imported videos tend to be high quality
            'StartCamera': 2.5,
            'Knowle': 2,
            'Web': 1.5,
            'LongImport': 1,
            'UNKNOWN': 0,
            'LongCamera': 0.5,
            'PictureSet': 0.5,
            'LongPicture': 0.5,
            'ACurlVideo': 0.5,
            'followShot': 0.5,
            'ShareFromOtherApp': 0.5,
            'SameFrame': 0,
            'PictureCopy': 0,
            'FlashPhoto': 0,
            'PhotoCopy': 0,
            'LocalCollection': 0,
            'LocalInteraction': 0
        }
        df["engagement_score"] += df['upload_type'].map(upload_type_weights).fillna(0)
    
    engagement_columns = {
        'like_cnt': 0.5,
        'comment_cnt': 0.7,
        'share_cnt': 0.8,
        'collect_cnt': 0.6,
        'follow_cnt': 0.9,
        'complete_play_cnt': 0.7,
        'valid_play_cnt': 0.5,
        'reply_comment_cnt': 0.6,
        'comment_like_cnt': 0.4
    }
    
    for col, weight in engagement_columns.items():
        if col in df.columns:
            df["engagement_score"] += np.minimum(np.log1p(df[col].fillna(0)) * weight, 10)


    penalty_columns = {
        'cancel_like_cnt': 0.4,
        'cancel_follow_cnt': 0.5,
        'report_cnt': 0.7
    }
    for col, weight in penalty_columns.items():
        if col in df.columns:
            df["engagement_score"] -= np.minimum(np.log1p(df[col].fillna(0)) * weight, 10)

    if 'video_width' in df.columns:
        df["engagement_score"] += np.where(df['video_width'] >= 720, 0.5, 0)

    if 'video_height' in df.columns:
        df["engagement_score"] += np.where(df['video_height'] >= 1280, 0.5, 0)
    
    # Here we normalize the score    
    min_score = df["engagement_score"].min()
    max_score = df["engagement_score"].max()

    df["engagement_score"] = (df["engagement_score"] - min_score) / (max_score - min_score)
    
    return df

test_df = build_engagement_score(test_df)
train_df = build_engagement_score(train_df)

## **3️⃣ Model Development**
📝 Associated Tasks :
- Choose a recommendation approach:
    - Collaborative filtering (e.g., ALS, Matrix Factorisation)
    - Content-based filtering
    - Sequence-aware models
    - Hybrid approaches
- Train and validate your model on the training set.

---

### ALS Model

The `engagement_score` we made in the previous section will act as the rating of our video. 

Therefore we can use ALS, which is a collaborative filtering model, that will try to guess the engagement score. The higher it is, the more engaging the video is.

The ALS Model was choosen here as it will take the different studied metrics in the one column `engagement_score` which will make more impact.

#### User item matrix 
First we will create the user-item matrix, which is essential for the ALS algorithm, to map the different users, videos and their engagement score.

In [16]:
# Get Unique user and videos
user_ids_train = train_df['user_id'].unique()
video_ids_train = train_df['video_id'].unique()

# Compute index for each user and videos
user_to_index = {user_id: idx for idx, user_id in enumerate(user_ids_train)}
video_to_index = {video_id: idx for idx, video_id in enumerate(video_ids_train)}

# add the index to the train and test
train_df['user_index'] = train_df['user_id'].map(user_to_index)
train_df['video_index'] = train_df['video_id'].map(video_to_index)

test_df['user_index'] = test_df['user_id'].map(user_to_index)
test_df['video_index'] = test_df['video_id'].map(video_to_index)

row = train_df['user_index'].values
col = train_df['video_index'].values

data = train_df['engagement_score'].values

n_users = train_df['user_index'].max() + 1
n_items = train_df['video_index'].max() + 1
    
user_item_matrix = csr_matrix((data, (row, col)), shape=(n_users, n_items))

#### Model Training
Here we train our ALS Model with the `user_item_matrix` 💪

The different parameters were found with the code you can find at the end of this notebook in the  "6️⃣ Trying to improve our model" section.

In [ ]:
model = AlternatingLeastSquares(
    factors=100,
    regularization=0.1,
    iterations=15,
    use_gpu=False,
    alpha=40
)

model.fit(user_item_matrix.T) 

/opt/homebrew/Caskroom/miniconda/base/lib/python3.12/site-packages/implicit/cpu/als.py:95: RuntimeWarning: OpenBLAS is configured to use 8 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
/opt/homebrew/Caskroom/miniconda/base/lib/python3.12/site-packages/implicit/utils.py:164: ParameterWarning: Method expects CSR input, and was passed csc_matrix instead. Converting to CSR took 0.05483412742614746 seconds
  warnings.warn(


  0%|          | 0/15 [00:00<?, ?it/s]

## **4️⃣ Recommendation Algorithm**
📝 Associated Tasks :
- Predict which videos are likely to be enjoyed by each user in the test set.
- Generate a top-N ranked list of recommendations for each user.

---
So let's define a simple function to get the top 100 recommendations for each user.

In [ ]:
top_n=100
def get_top_n_recommendations(model, user_item_matrix, user_ids, n=10, seen=True):
    recommendations = {}
    
    for user_id in user_ids:
        # Items the user has already interacted with (adjusted for training)
        already_interacted = set(user_item_matrix[user_id].indices) if seen else set()
        
        # U * V^T => computes the final score
        scores = model.user_factors[user_id].dot(model.item_factors.T)
       
        item_scores = [(item_id, scores[item_id])
                       for item_id in range(len(scores))
                       if item_id not in already_interacted]
        # Sort and select top-N items
        item_scores.sort(key=lambda x: x[1], reverse=True)
        top_items = [item[0] for item in item_scores[:n]]
        
        recommendations[user_id] = top_items

    return recommendations

train_users = train_df['user_index'].unique()
test_users = test_df['user_index'].unique()

# Exclude seen items (for real-world test set evaluation)
test_recommendations = get_top_n_recommendations(
    model, user_item_matrix, test_users, n=top_n, seen=True
)

# Include seen items (to understand model fit on training data)
train_recommendations = get_top_n_recommendations(
    model, user_item_matrix, train_users, n=top_n, seen=False
)

{14: [2267, 594, 122, 6755, 1399, 5767, 1515, 4262, 3591, 4409, 5198, 1829, 2763, 1121, 95, 5726, 4033, 1743, 1097, 7073, 5666, 4216, 6751, 6494, 6844, 2761, 3824, 4854, 6083, 1404, 525, 1131, 4665, 3472, 4917, 1938, 3005, 3730, 3253, 1488, 2830, 2925, 449, 1576, 3819, 1729, 4474, 3716, 3525, 6022, 2908, 4878, 6267, 5175, 5834, 618, 3524, 6785, 729, 6891, 5914, 2896, 5745, 2445, 4696, 2498, 5435, 5470, 2021, 733, 889, 5031, 2417, 1863, 3004, 4947, 170, 6352, 6941, 5748, 3438, 7102, 5392, 887, 5176, 5550, 303, 5064, 6616, 5993, 3743, 3752, 232, 4631, 713, 6697, 6482, 2171, 4314, 6857], 19: [419, 1404, 6308, 2928, 5536, 4181, 4259, 1169, 1455, 1421, 1068, 6726, 590, 5882, 2639, 2856, 6805, 6461, 2406, 958, 6034, 4401, 6996, 5543, 4212, 5411, 6514, 6498, 999, 4029, 3930, 1870, 492, 6300, 1582, 3692, 6289, 2608, 3716, 1214, 6, 5886, 4806, 382, 6577, 3347, 2332, 4376, 31, 5154, 4888, 2471, 2019, 4750, 5354, 3548, 2823, 4199, 5923, 3665, 2914, 593, 4314, 1614, 5645, 6282, 5214, 1204, 690, 63

In [25]:
    
def pretty_print_recommendations(test_recommendations, count=1):    
    for i, (user_id, items) in enumerate(test_recommendations.items()):
        print(f"User {user_id}: {items}")
        if i == count:
            break

pretty_print_recommendations(test_recommendations)    

User 14: [2267, 594, 122, 6755, 1399, 5767, 1515, 4262, 3591, 4409, 5198, 1829, 2763, 1121, 95, 5726, 4033, 1743, 1097, 7073, 5666, 4216, 6751, 6494, 6844, 2761, 3824, 4854, 6083, 1404, 525, 1131, 4665, 3472, 4917, 1938, 3005, 3730, 3253, 1488, 2830, 2925, 449, 1576, 3819, 1729, 4474, 3716, 3525, 6022, 2908, 4878, 6267, 5175, 5834, 618, 3524, 6785, 729, 6891, 5914, 2896, 5745, 2445, 4696, 2498, 5435, 5470, 2021, 733, 889, 5031, 2417, 1863, 3004, 4947, 170, 6352, 6941, 5748, 3438, 7102, 5392, 887, 5176, 5550, 303, 5064, 6616, 5993, 3743, 3752, 232, 4631, 713, 6697, 6482, 2171, 4314, 6857]
User 19: [419, 1404, 6308, 2928, 5536, 4181, 4259, 1169, 1455, 1421, 1068, 6726, 590, 5882, 2639, 2856, 6805, 6461, 2406, 958, 6034, 4401, 6996, 5543, 4212, 5411, 6514, 6498, 999, 4029, 3930, 1870, 492, 6300, 1582, 3692, 6289, 2608, 3716, 1214, 6, 5886, 4806, 382, 6577, 3347, 2332, 4376, 31, 5154, 4888, 2471, 2019, 4750, 5354, 3548, 2823, 4199, 5923, 3665, 2914, 593, 4314, 1614, 5645, 6282, 5214, 1204,

## **5️⃣ Evaluation**
📝 Associated Tasks :
- Choose suitable metrics (e.g., Precision@K, Recall@K, MAP, NDCG).
- Evaluate performance and provide interpretations.

---

For this section we will use several metrics to test our model, these are the metrics and their definitions that can be found in the course or with other ressources :
### 1. **Precision@K**

- Measures the **proportion of relevant items** among the top-K recommended items.

$$
\text{Precision@K} = \frac{\text{number of relevant recommended items in top-K}}{K}
$$

- **High Precision@K** → Users are more likely to find something they like within the first K items.  
- **Best for systems prioritizing “hit rate”** (e.g., e-commerce product suggestions).

--- 

### 2. **Mean Reciprocal Rank (MRR)**  
- Measures the **average rank** of the first relevant item across all queries. It’s especially useful for evaluating systems where the goal is to recommend one relevant item at the top.

$$
\text{MRR} = \frac{1}{|Q|} \sum_{q \in Q} \frac{1}{\text{rank}_q}
$$

Where:  
- Q is the set of queries or users.  
- rank_q is the rank of the first relevant item for query \( q \).

**Why use MRR?**  
- Evaluates how quickly the first relevant item appears. A higher MRR means the system performs better in bringing relevant results to the top.

---

### 3. **Hit Rate**  
- Measures the **proportion of users or queries** that have at least one relevant item in the top-K recommendations.

$$
\text{Hit Rate@K} = \frac{\text{number of queries with at least one relevant item in top-K}}{\text{total number of queries}}
$$

**Why use Hit Rate?**  
- Indicates how frequently the system successfully recommends at least one relevant item. A higher Hit Rate means more users are receiving relevant recommendations in the top-K results.
---

### 4. **Normalised Discounted Cumulative Gain (NDCG@K)**  

- Evaluates the quality of the ranking by giving **higher importance to top-ranked relevant items**.

    $$
    \text{DCG@K} = \sum_{i=1}^{K} \frac{rel_i}{\log_2(i + 1)}
    $$

    $$
    \text{NDCG@K} = \frac{\text{DCG@K}}{\text{IDCG@K}}
    $$

Where:  
- \( rel_i \) = relevance of item at position i (binary or graded relevance)  
- \( IDCG@K \) = DCG of an ideal ranking  

**Why use NDCG?**  
- Penalises relevant items that are **ranked lower**.  
- Captures the fact that users are more likely to click on **top-ranked items**.


In [ ]:
def evaluate_recommendations(recommendations, test_df, top_n=10, k=10):
    # Map of actual items per user
    user_actual_items = test_df.groupby('user_index')['video_index'].apply(set).to_dict()
    
    precision_at_n = []

    # Hit Rate, MRR, nDCG calculations
    hits = 0
    mrr = 0.0
    total_ndcg = 0.0
    count = 0
    
    for user_id, recommended_items in recommendations.items():
        if user_id in user_actual_items:
            actual_items = user_actual_items[user_id]
            recs_at_n = recommended_items[:top_n]

            # Precision
            num_relevant = len(set(recs_at_n) & actual_items)
            precision = num_relevant / len(recs_at_n) if recs_at_n else 0

            precision_at_n.append(precision)
            
            # Hit Rate
            gt = user_actual_items.get(user_id, set())
            if any(item in gt for item in recs_at_n[:k]):
                hits += 1

            # MRR
            for rank, item in enumerate(recs_at_n[:k], start=1):
                if item in gt:
                    mrr += 1.0 / rank
                    break

            # nDCG
            def dcg(recs, gt, k):
                return sum((1 / np.log2(i + 2)) if rec in gt else 0 for i, rec in enumerate(recs[:k]))

            def idcg(gt, k):
                n_relevant = min(len(gt), k)
                return sum(1 / np.log2(i + 2) for i in range(n_relevant))

            idcg_val = idcg(gt, k)
            if idcg_val > 0:
                total_ndcg += dcg(recs_at_n, gt, k) / idcg_val
                count += 1

    avg_precision = np.mean(precision_at_n) if precision_at_n else 0
    hit_rate = hits / len(user_actual_items) if user_actual_items else 0
    avg_mrr = mrr / len(user_actual_items) if user_actual_items else 0
    avg_ndcg = total_ndcg / count if count > 0 else 0

    return avg_precision, hit_rate, avg_mrr, avg_ndcg

print("Evaluating on training set...")
train_precision, train_hit_rate, train_mrr, train_ndcg = evaluate_recommendations(train_recommendations, train_df, top_n=top_n, k=10)
print(f"Training - Precision@{top_n}: {train_precision:.4f}, Hit Rate@{10}: {train_hit_rate:.4f}, MRR@{10}: {train_mrr:.4f}, nDCG@{10}: {train_ndcg:.4f}")


print("Evaluating on test set...")
test_precision, test_hit_rate, test_mrr, test_ndcg = evaluate_recommendations(test_recommendations, test_df, top_n=top_n, k=10)
print(f"Testing - Precision@{top_n}: {test_precision:.4f}, Hit Rate@{10}: {test_hit_rate:.4f}, MRR@{10}: {test_mrr:.4f}, nDCG@{10}: {test_ndcg:.4f}")

Evaluating on training set...
Training - Precision@100: 0.1967, Hit Rate@10: 0.7963, MRR@10: 0.3730, nDCG@10: 0.1992
Evaluating on test set...
Testing - Precision@100: 0.4611, Hit Rate@10: 1.0000, MRR@10: 0.6711, nDCG@10: 0.4662


### Interpretation

We observe that precision over 100 recommendations is lower on the training set than on the testing set, which could be due to matrix sparsity.

The training set, composed of `big_matrix.csv`, is much larger than the testing set, based on `small_matrix.csv`. 

As a result, there are more videos to recommend in the training set, which might impact precision.

- **Precision@100**  
  19% of the videos recommended in the training set and around 46% in the test set are relevant in the top 100 recommendations.

- **Hit Rate@10**  
  79% of users in the training set and 100% in the test set received at least one relevant item in their top 10 recommendations.

- **MRR@10**  
  The mean reciprocal rank for the top 10 recommendations is **0.37** on the training set and **0.67** on the test set.  
  On average, a relevant item appears around the **3rd or 4th** position in the top 10 for the training set, and around the **6th or 7th** position for the test set.

- **nDCG@10**  
  The normalized discounted cumulative gain at rank 10 is **0.1992** in the training set and **0.4662** in the test set.  
  This indicates that the top 10 recommendations are poorly ordered by relevance in the training set, but better ordered in the test set.

---

### Conclusion

- **Precision** is higher for this model on unseen data, suggesting good generalization.
- **Hit Rate** and **MRR** indicate the model consistently presents at least one relevant item in the top recommendations.
- **nDCG** shows that the model has **poor to moderate ranking quality**, especially in the training set.


## 6️⃣ Trying to Improve Our Model

The following code was used to test different parameter configurations of the **implicit ALS** model in order to find the one yielding the best **nDCG** score.

**nDCG** (Normalized Discounted Cumulative Gain) was chosen as the main evaluation metric because it was the most relevant for our context, for several reasons:

- **nDCG considers the position** of relevant items in the recommendation list.

- Since we recommend a short list of top videos to each user, it's important that the most relevant items appear at the top. nDCG is designed to prioritize this type of relevance.

- In real-world applications, users expect the best results early. If they have to scroll too long without finding relevant content, they may quickly lose interest. nDCG reflects this behavior well by penalizing lower-ranked relevant items.

> **In short**: nDCG provides a position-aware evaluation of the quality of our recommendations.

---

As the grid search took around 30 minutes to run, the corresponding code has been commented out for now. 

Moreover this experience was done in the [ALS notebook](models/als.ipynb) so it won't work here.

Below are the best metrics achieved by the model:

> **Best model:**  
> `{'factors': 100, 'regularization': 0.01, 'alpha': 40, 'hit_rate': 0.9978738483345145, 'mrr': 0.6703162791220928, 'ndcg': 0.46914296882680606}`


In [ ]:
"""
best_score = 0
best_params = {}

for factors in [20, 50, 100]:
    for reg in [0.01, 0.1, 0.5]:
        for alpha in [10, 40, 100]:
            print(f"\nTraining ALS with factors={factors}, reg={reg}, alpha={alpha}")
            model = AlternatingLeastSquares(
                factors=factors,
                regularization=reg,
                iterations=15,
                alpha=alpha,
                use_gpu=False
            )
            model.fit(user_item_matrix.T)

            test_recommendations = get_top_n_recommendations(
                model, user_item_matrix, test_users, n=top_n, seen=True
            )

            _, hit_rate, mrr, ndcg = evaluate_recommendations(
                test_recommendations, test_df, top_n=top_n, k=10
            )

            print(f"HitRate@10: {hit_rate:.4f}, MRR@10: {mrr:.4f}, nDCG@10: {ndcg:.4f}")

            if ndcg > best_score:
                best_score = ndcg
                best_params = {
                    'factors': factors,
                    'regularization': reg,
                    'alpha': alpha,
                    'hit_rate': hit_rate,
                    'mrr': mrr,
                    'ndcg': ndcg
                }

print(f"\nBest model: {best_params}")
"""



Training ALS with factors=20, reg=0.01, alpha=10


/opt/homebrew/Caskroom/miniconda/base/lib/python3.12/site-packages/implicit/utils.py:164: ParameterWarning: Method expects CSR input, and was passed csc_matrix instead. Converting to CSR took 0.0623021125793457 seconds
  warnings.warn(


  0%|          | 0/15 [00:00<?, ?it/s]

[(782, -0.06529804), (5006, -0.056622665), (4435, -0.056085333), (3353, -0.053640276), (2877, -0.053328965), (4898, -0.05258464), (167, -0.052474022), (6602, -0.050295018), (6944, -0.049785074), (3457, -0.0496398), (1231, -0.049157534), (2266, -0.048157725), (4181, -0.048140507), (4259, -0.046282977), (1235, -0.046027258), (740, -0.04548222), (3106, -0.045440182), (5783, -0.04529667), (2099, -0.044802938), (2902, -0.04379806), (2267, -0.043567643), (2128, -0.043005917), (4980, -0.042143233), (897, -0.04184061), (1997, -0.041838486), (6408, -0.041603338), (374, -0.041391682), (826, -0.04138065), (4069, -0.041363645), (4212, -0.04092903), (1356, -0.04084426), (5904, -0.04082241), (3681, -0.040590268), (2062, -0.040425476), (419, -0.04037657), (56, -0.039758198), (227, -0.03966455), (4768, -0.039439797), (1068, -0.039410677), (2915, -0.039250206), (5247, -0.038838256), (6910, -0.03858818), (6828, -0.03846199), (2072, -0.03811736), (1350, -0.038102634), (905, -0.03798085), (1325, -0.037845

/opt/homebrew/Caskroom/miniconda/base/lib/python3.12/site-packages/implicit/utils.py:164: ParameterWarning: Method expects CSR input, and was passed csc_matrix instead. Converting to CSR took 0.04934215545654297 seconds
  warnings.warn(


...

  0%|          | 0/15 [00:00<?, ?it/s]

[(4898, -0.44524014), (2069, -0.4060261), (6871, -0.36362535), (5284, -0.3438486), (6154, -0.33183503), (1235, -0.32393873), (6548, -0.31789577), (826, -0.3131903), (2925, -0.30954835), (6220, -0.30850115), (7013, -0.30509838), (1768, -0.30402285), (5893, -0.30060303), (60, -0.2997443), (59, -0.28919974), (4994, -0.2882564), (304, -0.284407), (1913, -0.28302848), (5035, -0.2817749), (7148, -0.28066757), (3344, -0.28011698), (2317, -0.27956697), (6591, -0.27939147), (6068, -0.27790046), (5641, -0.27645966), (1312, -0.27499795), (3404, -0.27110624), (5629, -0.2710791), (2038, -0.27107707), (1997, -0.26935577), (750, -0.26871765), (2574, -0.2672596), (7053, -0.2669547), (6293, -0.2656713), (1532, -0.26293272), (1202, -0.26153028), (6566, -0.25911492), (5319, -0.25850597), (1589, -0.25801724), (4774, -0.25708443), (6535, -0.25572085), (1169, -0.25412175), (4815, -0.2531256), (3377, -0.25108108), (2416, -0.24938722), (4991, -0.24879752), (6027, -0.24629961), (3963, -0.24604511), (2381, -0.2

/opt/homebrew/Caskroom/miniconda/base/lib/python3.12/site-packages/implicit/utils.py:164: ParameterWarning: Method expects CSR input, and was passed csc_matrix instead. Converting to CSR took 0.04874110221862793 seconds
  warnings.warn(


  0%|          | 0/15 [00:00<?, ?it/s]

[(4898, -0.6015213), (2925, -0.5640896), (3755, -0.5530015), (2924, -0.518721), (4942, -0.508617), (4991, -0.49120906), (1281, -0.48934618), (716, -0.48870647), (6951, -0.47688976), (4994, -0.4724596), (1629, -0.46617883), (6793, -0.46562603), (5705, -0.46290866), (1447, -0.4557876), (1064, -0.4526959), (2106, -0.45129174), (2693, -0.44904768), (5489, -0.447216), (2835, -0.44700035), (6566, -0.44608754), (2381, -0.44520396), (5756, -0.44452083), (1779, -0.4431992), (5284, -0.4412259), (4435, -0.43424818), (5413, -0.43277535), (626, -0.4308325), (1768, -0.4217257), (6048, -0.42119437), (4736, -0.41972998), (5423, -0.41920573), (4452, -0.41187137), (6904, -0.4110135), (795, -0.41054833), (1106, -0.40867117), (6113, -0.40696573), (6376, -0.4041444), (1178, -0.39760137), (2266, -0.3954322), (5631, -0.39412618), (1656, -0.39353287), (1203, -0.39263308), (2069, -0.39165547), (3521, -0.39136022), (2687, -0.39112014), (1220, -0.39005172), (390, -0.38808596), (3146, -0.3877945), (6910, -0.38699

You can now [go back to the main notebook](../FinalProject_Notebook.ipynb).